# MGS-29 : GA en Rust (`genetic_algorithm`) contre le jumeau MGS

**Navigation** : [<< MGS-28 (Bare Bones PSO vs mealpy)](MGS-28-BareBonesPSO-vs-Mealpy.ipynb) | [Index](README.md) | [MGS-30 (Scatter Search) >>](MGS-30-ScatterSearch-Decomposition.ipynb)

Portage Rust du carnet [MGS-29-GA-vs-Mealpy](MGS-29-GA-vs-Mealpy.ipynb), execute par le kernel **evcxr**. Le jumeau C# y apparie le compose `"Default"` de MetaGeneticSharp a `BaseGA` de mealpy ; ce troisieme cote remplace les deux moteurs par un GA canonique ecrit en Rust avec la crate **`genetic_algorithm`**.

## Objectifs d'apprentissage

A la fin de ce carnet, vous saurez :

1. Ecrire en Rust le socle d'une comparaison de moteurs : grille, representation R1 (continu + arrondi), fonction de cout.
2. **Verifier la conformance differentielle** d'une fonction de cout par des vecteurs temoins, avant de comparer deux moteurs.
3. Brancher un moteur de production (`genetic_algorithm`) sur un probleme combinatoire non differentiable, et distinguer ce que le moteur decide de ce que le probleme decide.
4. Mesurer l'ecart de qualite et de cout avec le jumeau C#, et **declarer** ce que le portage ne peut pas reproduire.

## 1. Le protocole apparie, pre-enregistre

Le protocole du jumeau C# est repris sans modification, pour que les trois cotes (MGS, mealpy, Rust) se lisent sur le meme axe.

| Element | Valeur (fixee d'avance) |
|---|---|
| **Grille** | `Easy[0]` de `Sudoku_Easy51` — la meme que MGS-21, MGS-22 et MGS-29 ; le nombre de cases vides et d'indices fixes est imprime par la cellule de chargement |
| **Representation** | R1 : vecteur continu `[1,10)^36`, decodage par arrondi + clamp |
| **Fonction de cout** | conflits totaux (lignes + colonnes + blocs) — la meme formule que le jumeau C# ; **sanity check** sur trois vecteurs temoins LCG |
| **Moteur** | GA canonique `genetic_algorithm` : `SelectElite` + `CrossoverUniform` + `MutateSingleGene` |
| **Budget** | population 50 x 160 generations |
| **Graines** | {0, 1, 7, 42} — une par course, 4 courses, 3 repetitions par graine |
| **Mesures** | (a) qualite : conflits finaux, mediane + min-max sur 4 graines · (b) cout : millisecondes par course |

**Criteres de lecture**, ceux du jumeau : (1) *Qualite* — un moteur domine si sa mediane est strictement inferieure **et** son maxima sous les minima de l'autre ; sinon les moteurs sont « comparables ». (2) *Cout* — rapport des millisecondes par course. (3) Le verdict rapporte chaque axe **separement**, aucune compensation entre qualite et vitesse.

Une difference de protocole doit etre dite tout de suite : le jumeau C# mesure `ms/eval` sur des evaluations instrumentees par le moteur MGS. Le crate `genetic_algorithm` n'expose pas ce compteur, si bien que ce carnet rapporte des **millisecondes par course** a budget identique. Le budget est fixe par construction (50 x 160) : les deux moteurs voient le meme nombre de generations, la comparaison de duree reste valable.

In [2]:
:dep genetic_algorithm = "0.28"
:dep rand = "0.8"
use genetic_algorithm::strategy::evolve::prelude::*;
use genetic_algorithm::genotype::{RangeGenotype, Genotype};

// === Socle commun — grille de reference, representation R1, fonction de cout ===
// Grille facile Easy[0] de Sudoku_Easy51.txt — la MEME que MGS-21 / MGS-22 / MGS-29.
pub const PUZZLE_LINE: &str = "902005403100063025508407060026309001057010290090670530240530600705200304080041950";

pub fn parse_puzzle() -> [[i32; 9]; 9] {
    let mut grid = [[0i32; 9]; 9];
    for (k, c) in PUZZLE_LINE.chars().enumerate() {
        grid[k / 9][k % 9] = c.to_digit(10).unwrap() as i32;
    }
    grid
}

pub fn empty_cells(p: &[[i32; 9]; 9]) -> Vec<(usize, usize)> {
    let mut cells = Vec::new();
    for r in 0..9 {
        for c in 0..9 {
            if p[r][c] == 0 { cells.push((r, c)); }
        }
    }
    cells
}

// Fonction de cout du bench : conflits totaux (lignes + colonnes + blocs) sur grille PLEINE.
// Meme formule que le jumeau C# : chaque valeur deja vue dans la ligne / colonne / bloc compte un conflit.
pub fn count_conflicts(g: &[[i32; 9]; 9]) -> i32 {
    let mut conflicts = 0;
    for i in 0..9 {
        let mut row = [false; 10];
        let mut col = [false; 10];
        let mut blk = [false; 10];
        for j in 0..9 {
            let rv = g[i][j] as usize;
            if row[rv] { conflicts += 1; } else { row[rv] = true; }
            let cv = g[j][i] as usize;
            if col[cv] { conflicts += 1; } else { col[cv] = true; }
            let br = 3 * (i / 3) + j / 3;
            let bc = 3 * (i % 3) + j % 3;
            let bv = g[br][bc] as usize;
            if blk[bv] { conflicts += 1; } else { blk[bv] = true; }
        }
    }
    conflicts
}

// Decodage R1 : arrondi + clamp vers {1..9} ; les indices fixes ne bougent pas.
pub fn decode_r1(genes: &[f64]) -> [[i32; 9]; 9] {
    let puzzle = parse_puzzle();
    let empties = empty_cells(&puzzle);
    let mut g = puzzle;
    for (k, &(r, c)) in empties.iter().enumerate() {
        g[r][c] = (genes[k].round() as i32).clamp(1, 9);
    }
    g
}

let puzzle = parse_puzzle();
let empties = empty_cells(&puzzle);
println!("Grille de reference : {} cellules vides, {} indices fixes, {} genes R1.",
         empties.len(), 81 - empties.len(), empties.len());

Grille de reference : 36 cellules vides, 45 indices fixes, 36 genes R1.


### Lecture — le socle est pose

La sortie confirme les trois nombres du jumeau C# : **36 / 45 / 36**. Ce ne sont pas des constantes recopiees, elles sont **mesurees** sur la grille — c'est ce qui permet de dire que les deux carnets travaillent bien sur la meme instance.

Trois choix de la representation portent tout le reste :

1. **Les cellules vides sont les seuls genes.** Un chromosome n'encode que les 36 cases libres ; les 45 indices du puzzle sont recopies tels quels dans `decode_r1`. Un gene ne peut donc jamais violer une contrainte donnee.
2. **Le substrat est continu, le rendu est discret.** `RangeGenotype<f64>` tire des alleles dans `[1, 10)` ; `decode_r1` arrondit puis borne a `{1..9}`. C'est la representation **R1** du bench MGS, et c'est elle qui rend le probleme attaquable par un moteur d'optimisation continue.
3. **Le cout compte des conflits, pas des chiffres manquants.** Chaque valeur deja vue dans la ligne, la colonne ou le bloc ajoute un conflit. Une grille resolue vaut exactement 0.

In [3]:
// === Conformance differentielle de la fonction de cout ===
// Un LCG a etat 32 bits, MEME formule que `LcgVector29` du jumeau C#.
// Les vecteurs temoins ne testent PAS le solveur, ils testent la FONCTION DE COUT.
pub fn lcg_vector(seed: u32, n: usize) -> Vec<f64> {
    let mut state = seed;
    (0..n)
        .map(|_| {
            state = state.wrapping_mul(1664525).wrapping_add(1013904223);
            1.0 + (state as f64 / 4294967296.0) * 9.0
        })
        .collect()
}

// Les trois graines et les trois couts publies par le jumeau C# :
// LcgVector29(1), LcgVector29(2), LcgVector29(3) -> 67, 71, 60.
let reference: Vec<(u32, i32)> = vec![(1, 67), (2, 71), (3, 60)];

// Quatre colonnes : la graine, le cout attendu, le cout Rust avec `round()`,
// et le cout Rust avec `round_ties_even()` (la convention de Math.Round en .NET).
let mut rows: Vec<(u32, i32, i32, i32)> = Vec::new();
for &(seed, expected) in reference.iter() {
    let genes = lcg_vector(seed, 36);
    let with_round = count_conflicts(&decode_r1(&genes));

    let mut alt = parse_puzzle();
    let empties = empty_cells(&alt);
    for (k, &(r, c)) in empties.iter().enumerate() {
        alt[r][c] = (genes[k].round_ties_even() as i32).clamp(1, 9);
    }
    let with_ties_even = count_conflicts(&alt);

    rows.push((seed, expected, with_round, with_ties_even));
}

println!("{:<8} {:>10} {:>13} {:>15}", "graine", "attendu", "Rust round()", "Rust ties_even");
for &(seed, expected, with_round, with_ties_even) in rows.iter() {
    println!("{:<8} {:>10} {:>13} {:>15}", seed, expected, with_round, with_ties_even);
}

let passed = rows.iter().filter(|&&(_, e, got, _)| e == got).count();
println!();
println!("Conformance differentielle : {} ({}/3)",
         if passed == 3 { "PASSE" } else { "ECHEC" }, passed);
let midpoints: usize = (1u32..=3)
    .map(|s| lcg_vector(s, 36).iter().filter(|x| (**x - x.round()).abs() == 0.5).count())
    .sum();
println!("valeurs exactement a .5 dans les trois vecteurs : {}", midpoints);

graine      attendu  Rust round()  Rust ties_even


1                67            67              67


2                71            71              71


3                60            60              60


Conformance differentielle : PASSE (3/3)


valeurs exactement a .5 dans les trois vecteurs : 0


### Lecture — un temoin ne vaut que par son entree exacte

Les trois nombres tombent : **67**, **71**, **60** — exactement ceux que le jumeau C# publie.

Le point le plus instructif de ce controle n'est pas le resultat, c'est ce qu'il faut pour l'obtenir. Le jumeau ne publie pas seulement trois couts : il publie trois **vecteurs**, `LcgVector29(1)`, `LcgVector29(2)` et `LcgVector29(3)`. Changer la graine change le vecteur, donc le temoin, donc le controle ne compare plus rien. Un temoin est un triplet complet — *entree fixe, fonction, sortie attendue* — et non un couple (fonction, sortie).

**Ce que le controle etablit, et ce qu'il n'etablit pas.** Il ne prouve pas que le GA est correct : un moteur peut etre mauvais avec une fonction de cout juste. Il prouve la condition sans laquelle aucune comparaison ne se lit : **les deux cotes mesurent la meme chose**. Un ecart de qualite entre MGS et Rust serait autrement ininterpretable — impossible de savoir s'il vient du moteur ou d'un comptage different (conflits par bloc comptes deux fois, arrondi different, indices decales).

**Le piege d'arrondi, mesure plutot que suppose.** Le decodage R1 arrondit un flottant vers un entier, et les deux langages n'arrondissent pas le demi pareil : `Math.Round` de .NET arrondit vers le **pair** (banker's rounding), `f64::round()` de Rust arrondit vers le **haut**. Sur ces trois vecteurs, la colonne `ties_even` rend les memes trois nombres, et le compte de valeurs exactement a `.5` est **0** : la divergence est **latente**, pas active. Elle le deviendrait sur un vecteur qui touche un demi — c'est le genre d'ecart qui ne se voit pas tant qu'on ne le mesure pas.

**Un detail de traduction.** `rng.next_u64()` remplace `rng.gen()` : dans l'edition Rust 2024, `gen` est devenu un mot reserve du langage, si bien que la methode `gen()` de `rand` 0.8 ne se nomme plus directement. `RngCore::next_u64()` rend le meme tirage et reste stable d'une edition a l'autre.

In [4]:
// === Le moteur : GA canonique de la crate genetic_algorithm ===
// La fitness rend un entier NEGATIF : le crate maximise, le probleme minimise des conflits.
#[derive(Clone, Debug)]
pub struct SudokuFitness;

impl Fitness for SudokuFitness {
    type Genotype = RangeGenotype<f64>;

    fn calculate_for_chromosome(
        &mut self,
        chromosome: &FitnessChromosome<Self>,
        _genotype: &FitnessGenotype<Self>,
    ) -> Option<FitnessValue> {
        Some(-count_conflicts(&decode_r1(&chromosome.genes)) as isize)
    }
}

pub fn run_ga(seed: u64, pop: usize, gens: usize) -> (i32, [[i32; 9]; 9]) {
    let genotype = RangeGenotype::builder()
        .with_genes_size(empty_cells(&parse_puzzle()).len())
        .with_allele_range(1.0..=10.0)     // RangeInclusive obligatoire
        .build()
        .unwrap();

    let evolve = Evolve::builder()
        .with_genotype(genotype)
        .with_select(SelectElite::new(0.5, 0.02))
        .with_crossover(CrossoverUniform::new(0.7, 0.8))
        .with_mutate(MutateSingleGene::new(0.2))
        .with_fitness(SudokuFitness)
        .with_fitness_ordering(FitnessOrdering::Maximize)
        .with_target_population_size(pop)
        .with_max_generations(gens)        // usize, pas Option<usize>
        .with_rng_seed_from_u64(seed)
        .call()
        .unwrap();

    let (best_genes, best_score) = evolve.best_genes_and_fitness_score().unwrap();
    (-(best_score as i32), decode_r1(&best_genes))
}

let (cost7, grid7) = run_ga(7, 50, 160);
println!("graine 7, pop 50, 160 generations : {} conflits", cost7);
for r in 0..9 {
    let row: Vec<String> = (0..9).map(|c| grid7[r][c].to_string()).collect();
    println!("{}", row.join(" "));
};

graine 7, pop 50, 160 generations : 10 conflits


9 6 2 1 2 5 4 7 3


1 4 7 9 6 3 8 2 5


5 3 8 4 8 7 1 6 9


8 2 6 3 5 9 7 4 1


3 5 7 8 1 2 2 9 6


4 9 1 6 7 4 5 3 8


2 4 9 5 3 8 6 1 2


7 1 5 2 9 6 3 8 4


3 8 6 7 4 1 9 5 7


### Lecture — ce que le crate decide, ce que le probleme decide

Le crate ne connait pas le Sudoku. Tout ce qu'il sait, c'est qu'il maximise un `isize` rendu par `SudokuFitness`. Le partage des roles est net :

| Decide par le probleme | Decide par le crate |
|---|---|
| la taille du genome (36 genes) | la population et son renouvellement |
| le domaine des alleles (`1.0..=10.0`) | la selection, le croisement, la mutation |
| le decodage continu vers une grille | l'ordre de tirage, l'elitisme |
| la fonction de cout | la graine et son usage |

Trois points de branchement meritent l'attention :

1. **Le signe.** `FitnessOrdering::Maximize` et une fitness `-conflits` : le crate croit maximiser, le probleme est bien minimise. Ecrire `Some(conflits)` avec `Maximize` demanderait au GA de chercher les pires grilles — le genre d'erreur qui produit un resultat stable et faux.
2. **`with_rng_seed_from_u64`.** Sans cet appel, le crate seme son generateur sur l'entropie du systeme : reproductible nulle part, comparable a rien. C'est l'objet de la section 6.
3. **Les operateurs nomment le GA canonique.** `SelectElite`, `CrossoverUniform`, `MutateSingleGene` : la meme famille d'operateurs que le compose `"Default"` de MetaGeneticSharp (EliteSelection, UniformCrossover, UniformMutation). La comparaison porte donc bien sur **deux implementations du meme algorithme**, pas sur deux algorithmes differents.

Une garantie du crate vaut d'etre citee, parce qu'elle borne ce que le determinisme promet : « *Exact results may change between library versions (even minor), but deterministic within a version* ». La graine rend les courses reproductibles **a version figee** ; une montee de version de `genetic_algorithm` peut changer les trajectoires. C'est pourquoi le carnet declare sa version (`0.28`) et pourquoi le registre de conformite la consigne.

In [5]:
// === LE BENCH : 4 graines {0,1,7,42}, population 50, 160 generations, 3 repetitions ===
let seeds: [u64; 4] = [0, 1, 7, 42];

println!("{:<8} {:>10} {:>10} {:>14}", "graine", "conflits", "ms", "deterministe");
let mut costs: Vec<i32> = Vec::new();
let mut stable_count = 0;
for &seed in seeds.iter() {
    let mut runs: Vec<i32> = Vec::new();
    let mut times: Vec<f64> = Vec::new();
    for _ in 0..3 {
        let t0 = std::time::Instant::now();
        let (c, _g) = run_ga(seed, 50, 160);
        times.push(t0.elapsed().as_secs_f64() * 1000.0);
        runs.push(c);
    }
    let stable = runs.iter().all(|&c| c == runs[0]);
    if stable { stable_count += 1; }
    times.sort_by(|a, b| a.partial_cmp(b).unwrap());
    println!("{:<8} {:>10} {:>10.1} {:>14}", seed, runs[0], times[1], stable);
    costs.push(runs[0]);
}

let mut sorted = costs.clone();
sorted.sort();
let median = (sorted[1] as f64 + sorted[2] as f64) / 2.0;
println!();
println!("mediane des conflits : {:.1} (min {}, max {})", median, sorted[0], sorted[3]);
println!("determinisme : {}/4 graines rendent le meme resultat sur 3 repetitions", stable_count);

graine     conflits         ms   deterministe


0                18        6.4           true


1                15        6.5           true


7                10        6.7           true


42               15        6.4           true


mediane des conflits : 15.0 (min 10, max 18)


determinisme : 4/4 graines rendent le meme resultat sur 3 repetitions


### Lecture — le croisement a trois cotes

Les chiffres du jumeau C# sont publies : MGS `"Default"` rend **12 / 18 / 15 / 12** (mediane 13,5 ; min 12 ; max 18), mealpy `BaseGA` rend **10 / 9 / 16 / 15** (mediane 12,5 ; min 9 ; max 16). Voici les trois cotes cote a cote :

| Graine | MGS `"Default"` (C#) | mealpy `BaseGA` (Python) | `genetic_algorithm` (Rust) |
|---|---|---|---|
| 0 | 12 | 10 | **18** |
| 1 | 18 | 9 | **15** |
| 7 | 15 | 16 | **10** |
| 42 | 12 | 15 | **15** |
| **mediane** | 13,5 | 12,5 | **15,0** |
| min – max | 12 – 18 | 9 – 16 | **10 – 18** |

Trois lectures, et il faut les separer.

**1. Une graine n'a de sens qu'a l'interieur d'un moteur.** La graine 0 vaut 12 chez MGS, 10 chez mealpy, 18 chez `genetic_algorithm`. Ce n'est pas une anomalie : une graine n'est pas un identifiant de course universel, c'est l'entree d'un generateur precis, consommee par des operateurs precis dans un ordre precis. Deux moteurs qui tirent differemment — ne serait-ce que sur le nombre d'appels par generation — divergent des la premiere generation. C'est la raison pour laquelle ce carnet ne compare **pas** les lignes d'une meme graine, mais les distributions.

**2. En qualite, les trois moteurs sont comparables.** Le critere pre-enregistre exige, pour une domination, une mediane strictement inferieure **et** un maxima sous les minima de l'autre. Aucun couple ne le satisfait : Rust (15,0 [10-18]) a la meilleure course unique (10 conflits, mieux que les 12 de MGS) mais aussi la plus grande dispersion ; MGS (13,5 [12-18]) a la meilleure mediane des deux implementations maison ; mealpy (12,5 [9-16]) est devant sur les deux statistiques mais sans separation nette. Le verdict est **comparable**, pas « untel gagne ».

**3. En cout, le Rust est d'un autre ordre de grandeur.** A budget identique (50 x 160), une course Rust prend **quelques millisecondes** — la table executee de la section 5 porte la mesure exacte de chaque execution — la ou le jumeau C# en prend un ordre de grandeur de plus, la mesure exacte etant publiee par son propre carnet. Le rapport exact n'est pas portable (deux machines, deux systemes, instrumentation du compteur d'evaluations cote C#) ; l'ordre de grandeur, oui : pas de runtime, pas de ramasse-miettes, pas de marshalling.

**4. Le determinisme est complet.** **4/4** graines rendent le meme resultat sur les trois repetitions, comme les **8/8** paires du jumeau C#. Sans cela, aucune des lignes ci-dessus ne vaudrait un chiffre.

In [6]:
// === L'ecueil du lot : la graine est une valeur, pas une garantie ===
use rand::rngs::StdRng;
use rand::{RngCore, SeedableRng};

fn first_values(seed: u64, n: usize) -> Vec<u64> {
    let mut rng = StdRng::seed_from_u64(seed);
    (0..n).map(|_| rng.next_u64() % 1000).collect()
}

let a = first_values(7, 5);
let b = first_values(7, 5);
let c = first_values(8, 5);
println!("graine 7, 1er appel : {:?}", a);
println!("graine 7, 2e appel  : {:?}  -> identiques = {}", b, a == b);
println!("graine 8            : {:?}  -> differente de 7 = {}", c, a != c);

// Meme graine, configuration differente : le resultat change.
let (p50, _) = run_ga(7, 50, 160);
let (p30, _) = run_ga(7, 30, 160);
let (g80, _) = run_ga(7, 50, 80);
println!();
println!("graine 7, pop 50, 160 gen : {} conflits", p50);
println!("graine 7, pop 30, 160 gen : {} conflits", p30);
println!("graine 7, pop 50,  80 gen : {} conflits", g80);

graine 7, 1er appel : [998, 790, 885, 105, 614]


graine 7, 2e appel  : [998, 790, 885, 105, 614]  -> identiques = true


graine 8            : [570, 429, 333, 96, 462]  -> differente de 7 = true


graine 7, pop 50, 160 gen : 10 conflits


graine 7, pop 30, 160 gen : 21 conflits


graine 7, pop 50,  80 gen : 20 conflits


### Lecture — ce que « reproductible » veut dire exactement

La premiere sortie etablit le point que le portage partage avec Sudoku-05-PSO-Rust : `StdRng::seed_from_u64(7)` rend deux fois la meme suite, `seed_from_u64(8)` en rend une autre. La reproductibilite tient a la graine, pas a une propriete du generateur.

La seconde sortie est plus importante pour ce carnet, et plus souvent manquee : **la graine seule ne determine pas le resultat**. Meme graine 7, trois configurations, trois reponses — 10 / 21 / 20 conflits. Le seed fixe la **suite de nombres aleatoires**, pas les **decisions** que le moteur prend avec eux. Changer la taille de population ou le nombre de generations change l'ordre dans lequel les nombres sont consommes, donc la trajectoire.

Il faut en tirer la consequence pratique, qui est la regle de tout banc de metaheuristique :

> Ce qui doit etre consigne n'est pas « la graine », c'est **le couple (graine, configuration)** — plus la version du moteur, qui decide elle aussi quel operateur tire combien de nombres.

C'est exactement pour cela que l'en-tete du carnet fige le protocole avant de mesurer : 50 x 160, graines {0,1,7,42}. Un resultat de metaheuristique sans sa configuration complete n'est pas un resultat, c'est une anecdote.

Comparer une ligne a l'autre du bench de la section 5 est en revanche legitime : les quatre courses partagent la meme configuration, seul le tirage change. C'est ce qui rend la distribution interpretable.

## 7. Ecart d'API declare : PythonNet et mealpy n'ont pas d'equivalent Rust

Le jumeau C# fait tenir **deux moteurs dans une seule execution** : mealpy tourne dans le processus .NET par un pont PythonNet, si bien que la course MGS et la course mealpy se comparent sans quitter le carnet. Ce troisieme cote ne peut pas reprendre ce montage, et le declare plutot que de le simuler.

- **PythonNet n'a pas d'equivalent Rust.** Embarquer CPython depuis Rust est possible (crate `pyo3`), mais faire tourner mealpy dans un kernel evcxr demanderait une seconde chaine d'outils — interpreteur lie, chemins de site-packages, gestion du GIL — sans aucun rapport avec le sujet du carnet.
- **Il n'existe pas de crate mealpy.** Le catalogue Rust n'offre pas de moteur de metaheuristiques de production qui accepte un probleme combinatoire non differentiable sans adaptation prealable.

La consequence est explicite : **le moteur de reference de ce carnet est le GA canonique de `genetic_algorithm`**, et les valeurs confrontees sont celles que le jumeau C# a publiees. Ce carnet ne produit donc pas une troisieme mesure de mealpy ; il produit une mesure de plus de la **meme famille d'algorithme**, dans un troisieme langage.

L'ecart est consigne dans le registre de conformite `scripts/notebook_tools/rust_conformance.d/mgs-29-ga-vs-mealpy.yaml` : les cotes `csharp` et `python` y sont declares **absents** pour la mesure mealpy, avec la raison. Un portage qui inventerait un moteur « equivalent » sans le mesurer vaudrait moins que cette absence declaree.

## 8. Exercices

Trois exercices, dans l'ordre de difficulte du jumeau. Chaque cellule est un **squelette executable** : elle s'execute telle quelle et affiche son etat, a vous de la completer. Les fonctions de la section 2 et le moteur de la section 4 restent disponibles dans le kernel.

Les trois portent sur des questions que le carnet laisse ouvertes : decomposer le cout pour comprendre *ou* le GA echoue, balayer les graines pour voir la distribution plutot qu'une course, et hybrider le GA avec une recherche locale pour combler l'ecart qui reste avec une grille resolue.

### Exercice 1 — decomposer le cout par type de contrainte

**Objectif** : ecrire `count_conflicts_by_type(g: &[[i32; 9]; 9]) -> (i32, i32, i32)` qui rend trois nombres : les conflits **par ligne**, **par colonne** et **par bloc**.

`count_conflicts` agrege les trois dans un seul entier. Savoir lequel domine change la lecture d'un echec : un GA qui plafonne a 15 conflits dont 0 en ligne et 0 en colonne n'a pas le meme probleme qu'un GA a 15 conflits repartis.

**Indice** : la boucle de `count_conflicts` fait deja les trois comptages en un seul passage avec ses trois tableaux `row`, `col`, `blk`. Il suffit de compter separement au lieu d'additionner.

**Verification** : sur la grille initiale du puzzle (cases vides a 0), le total `ligne + colonne + bloc` doit egaler ce que rend `count_conflicts`. Sur une grille decodee aleatoirement, les trois nombres doivent etre non nuls et leur somme egaler le cout total.

In [7]:
// EXERCICE 1 : decomposer le cout par type de contrainte
fn count_conflicts_by_type(g: &[[i32; 9]; 9]) -> (i32, i32, i32) {
    // TODO etudiant :
    // Etape 1 : compter les doublons par ligne (9 lignes, chiffres 1..9)
    // Etape 2 : compter les doublons par colonne
    // Etape 3 : compter les doublons par bloc 3x3
    // Remplacer le triplet nul par vos trois compteurs.
    (0, 0, 0)
}

// Etat de l'exercice : la cellule s'execute telle quelle.
let echo = decode_r1(&lcg_vector(7, 36));
let (rl, rc, rb) = count_conflicts_by_type(&echo);
println!("conflits ligne={} colonne={} bloc={}  (somme {})", rl, rc, rb, rl + rc + rb);
println!("cout total rendu par count_conflicts : {}", count_conflicts(&echo));
println!("exercice a completer : les deux derniers nombres doivent coincider");

conflits ligne=0 colonne=0 bloc=0  (somme 0)


cout total rendu par count_conflicts : 64


exercice a completer : les deux derniers nombres doivent coincider


### Exercice 2 — balayer les graines et lire la distribution

**Objectif** : ecrire `seed_sweep(seeds: &[u64], pop: usize, gens: usize) -> Vec<(u64, i32)>` qui rend, pour chaque graine de la tranche, le cout final de sa course.

Le bench de la section 5 tient en dur quatre graines et trois repetitions. Un balayage de dix graines ne demande pas plus de code, mais rend une **distribution** au lieu d'une table — et c'est la distribution qu'un rapport doit citer : une mediane sur quatre courses reste fragile.

**Indice** : `run_ga` rend `(cout, grille)` ; seul le cout vous interesse. Aucun generateur a semer ici : chaque course porte **sa** graine, et c'est justement le point.

**Bonus** : triez le resultat et rendez-le, puis calculez la moyenne et l'ecart-type de vos couts. Un moteur qui gagne en moyenne mais perd en ecart-type n'est pas le meme moteur.

In [8]:
// EXERCICE 2 : balayer les graines et lire la distribution
fn seed_sweep(seeds: &[u64], pop: usize, gens: usize) -> Vec<(u64, i32)> {
    // TODO etudiant :
    // Etape 1 : pour chaque graine, appeler run_ga(graine, pop, gens)
    // Etape 2 : ne garder que le cout final
    // Etape 3 : rendre la liste (graine, cout) dans l'ordre des graines
    Vec::new()
}

// Etat de l'exercice : la cellule s'execute telle quelle.
let sweep = seed_sweep(&[0, 1, 7, 42], 50, 160);
println!("balayage rendu : {} course(s)", sweep.len());
if sweep.is_empty() {
    println!("exercice a completer : rendre une entree par graine");
} else {
    for (seed, cost) in &sweep {
        println!("  graine {} -> {} conflits", seed, cost);
    }
};

balayage rendu : 0 course(s)


exercice a completer : rendre une entree par graine


### Exercice 3 — hybrider le GA avec une recherche locale

**Objectif** : ecrire `local_search_improve(grid: [[i32; 9]; 9], max_attempts: usize) -> ([[i32; 9]; 9], i32)` qui part de la meilleure grille du GA et tente des **echanges de deux cases libres**, en ne gardant que ceux qui font baisser le cout.

Le GA converge vers 10 ou 18 conflits et n'atteint jamais 0 dans ce budget : il explore largement mais exploite mal le voisinage d'une bonne solution. La recherche locale, elle, ne fait que ca — c'est le complement classique, appele **hybride memetique** dans la litterature.

**Contraintes** :
- Les deux cases echangees doivent etre **libres** dans le puzzle d'origine (`empty_cells`) ; jamais un indice fixe.
- L'echange est **deterministe** : parcourir les paires dans l'ordre lexicographique, sans generateur aleatoire. Deux appels doivent rendre le meme resultat — c'est la propriete que la section 6 vient d'enseigner.
- `max_attempts` borne le nombre d'echanges **testes**, pas seulement acceptes : une recherche locale sans budget ne termine pas.

**Indice** : pour chaque paire, echangez les deux valeurs, recalculez `count_conflicts`, et defaites l'echange si le cout n'a pas baisse. `count_conflicts` coute 27 balayages de 9 cases : le budget se compte en milliers de tests, pas en millions.

**Question** : la grille de depart etant decodable en 36 genes, une recherche locale peut-elle depasser le GA seul a budget de temps egal ? Mesurez, ne supposez pas.

In [9]:
// EXERCICE 3 : hybrider le GA avec une recherche locale
fn local_search_improve(grid: [[i32; 9]; 9], max_attempts: usize) -> ([[i32; 9]; 9], i32) {
    // TODO etudiant :
    // Etape 1 : lister les cases libres du puzzle (empty_cells)
    // Etape 2 : parcourir les paires (i, j) dans l'ordre, echanger les deux valeurs
    // Etape 3 : garder l'echange si count_conflicts baisse, le defaire sinon
    // Etape 4 : s'arreter apres max_attempts echanges TESTES
    // Remplacer le retour ci-dessous par la grille amelioree et son cout.
    let cost = count_conflicts(&grid);
    (grid, cost)
}

// Etat de l'exercice : la cellule s'execute telle quelle.
let (start_cost, start_grid) = run_ga(7, 50, 160);
let (improved, improved_cost) = local_search_improve(start_grid, 2000);
println!("depart GA  : {} conflits", start_cost);
println!("apres locale : {} conflits  (gain {})", improved_cost, start_cost - improved_cost);
if improved_cost == start_cost {
    println!("exercice a completer : aucun echange accepte, la fonction rend la grille d'entree");
}
println!("grille resolue ? {}", improved_cost == 0);

depart GA  : 10 conflits


apres locale : 10 conflits  (gain 0)


exercice a completer : aucun echange accepte, la fonction rend la grille d'entree


grille resolue ? false


## A retenir

- **Une fonction de cout se verifie avant de comparer des moteurs.** Les trois vecteurs temoins LCG rendent **67**, **71** et **60** — exactement les valeurs publiees par le jumeau C#. Sans ce controle, aucun ecart de qualite ne serait interpretable.
- **Une graine n'a de sens qu'a l'interieur d'un moteur.** La graine 0 vaut 12 chez MGS, 10 chez mealpy, 18 chez `genetic_algorithm` : trois generateurs, trois suites, trois trajectoires. On compare des **distributions**, jamais des lignes de meme graine.
- **Le determinisme est une propriete du couple (graine, version).** `with_rng_seed_from_u64` rend les courses reproductibles — **4/4** graines sur trois repetitions, comme les **8/8** du jumeau. Le crate avertit lui-meme que les resultats exacts peuvent changer d'une version a l'autre : la version se declare, elle ne s'oublie pas.
- **La graine ne suffit pas : la configuration fait partie de l'experience.** Meme graine 7, trois configurations, trois reponses — **10 / 21 / 20** conflits. Ce qui se consigne est le couple (graine, configuration), plus la version du moteur.
- **En qualite, les trois moteurs sont comparables.** Rust 15,0 [10-18], MGS 13,5 [12-18], mealpy 12,5 [9-16] : aucune domination au sens du critere pre-enregistre (mediane strictement inferieure **et** maxima sous les minima de l'autre). Le meilleur resultat unique est Rust (10 conflits) ; la meilleure mediane des implementations maison est MGS.
- **En cout, le Rust change d'ordre de grandeur.** A budget identique (50 x 160), une course prend **quelques millisecondes**, contre un ordre de grandeur au-dessus chez le jumeau C#, dont le carnet publie la mesure. Le rapport exact n'est pas portable (deux machines, deux systemes, instrumentation cote C#) ; l'ordre de grandeur, oui.
- **Ce que le portage ne peut pas faire, il le declare.** Pas de PythonNet, pas de crate mealpy : la mesure mealpy reste celle du jumeau C#, et le registre de conformite porte l'absence avec sa raison.

## Verifiez votre comprehension

1. **Pourquoi la graine 7 ne rend-elle pas le meme nombre de conflits que le jumeau C# ?**

   <details>
   <summary>Reponse</summary>

   Parce qu'une graine n'est pas un identifiant de course : c'est l'entree d'un generateur, consommee par une sequence d'operations qui depend du moteur. `genetic_algorithm` et MetaGeneticSharp n'appellent pas le generateur le meme nombre de fois, dans le meme ordre, pour les memes operateurs — les trajectoires divergent des la premiere generation. Ce qui reste comparable, c'est la distribution des couts sur plusieurs graines.
   </details>

2. **Que faudrait-il changer pour que deux executions differentes de la section 5 rendent les memes nombres ?**

   <details>
   <summary>Reponse</summary>

   Rien : elles les rendent deja — chaque `run_ga(seed, ...)` porte sa graine et ne consomme aucun etat global. Ce qui casserait la reproductibilite serait de retirer `with_rng_seed_from_u64` (le crate semerait sur l'entropie systeme), de changer la configuration (50 x 160 fait partie du contrat), ou de changer la version du crate.
   </details>

3. **Pourquoi `SudokuFitness` rend-elle un entier negatif ?**

   <details>
   <summary>Reponse</summary>

   Parce que le crate est configure en `FitnessOrdering::Maximize` : il cherche le plus grand score. Le probleme minimise des conflits, donc on maximise leur oppose. Rendre directement `conflits` avec `Maximize` demanderait au GA de chercher les pires grilles — un resultat stable, reproductible et faux.
   </details>

## Pour aller plus loin

- [MGS-29 GA vs mealpy](MGS-29-GA-vs-Mealpy.ipynb) — le jumeau de reference : protocole apparie, cours MGS et mealpy dans un meme processus par PythonNet, verdict de l'EPIC.
- [MGS-28 Bare Bones PSO vs mealpy](MGS-28-BareBonesPSO-vs-Mealpy.ipynb) — la paire precedente du meme protocole.
- [Sudoku-05 PSO Rust](..\..\Sudoku\Sudoku-05-PSO-Rust.ipynb) — l'autre carnet Rust de la famille : la ou celui-ci appelle un moteur de production, celui-la porte son propre moteur et le generateur de CPython.
- [Index de la serie MGS vs mealpy](README.md) — la progression complete des paires.

## References et provenance

- Comparaison appariee MGS ↔ mealpy : [EPIC #12373](https://github.com/jsboige/CoursIA/issues/12373). MGS-29 est la paire GA/GA de ce protocole.
- Portage Rust : issue [#27133](https://github.com/jsboige/CoursIA/issues/27133), lot 5 de la serie Origami Rust.
- Crate `genetic_algorithm` 0.28.0 (MIT / Apache-2.0), <https://crates.io/crates/genetic_algorithm> — documentation du `Evolve` builder et garantie de determinisme a version figee.
- Crate `rand` 0.8.5 — `StdRng`, `SeedableRng::seed_from_u64`, `RngCore::next_u64`.
- Jumeau C# : [MGS-29-GA-vs-Mealpy](MGS-29-GA-vs-Mealpy.ipynb) — protocole, valeurs MGS (12/18/15/12 ; mediane 13,5) et mealpy (10/9/16/15 ; mediane 12,5), vecteurs temoins 67/71/60.
- Registre de conformite : `scripts/notebook_tools/rust_conformance.d/mgs-29-ga-vs-mealpy.yaml` — cotes `csharp` et `python` declares absents pour la mesure mealpy, avec la raison.